# Stage 2.6 Training — `v2_mapless_wa` (NVIDIA L4, BF16, CUDA Flash)

1. Confirm runtime is **L4 GPU** (`Runtime -> Change runtime type -> L4 GPU`).
2. Click **`Runtime -> Run all` (`Ctrl+F9`)** and approve the Google Drive mount prompt.
3. Everything else (parallel SHA-256 verify + unpack to `/content/corpus`, 1-step smoke check, 10-epoch training, 60s Drive backup + `heartbeat.json`, final SHA-256 verification, and `runtime.unassign()` auto-shutdown) runs unattended.

In [ ]:
# %% 1. GPU Check + Mount Drive + Install Deps
import importlib
import shutil
import subprocess
import sys
import torch
from pathlib import Path

smi = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    text=True,
).strip() if shutil.which('nvidia-smi') else 'NO_GPU'
print(f'GPU: {smi}')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not detected! Switch Runtime -> Change runtime type -> L4 GPU, then Run all.')
dev_name = torch.cuda.get_device_name(0)
bf16_ok = torch.cuda.is_bf16_supported()
disk_gb = shutil.disk_usage('/content').free / (1024 ** 3)
print(f'torch={torch.__version__} | device={dev_name} | bf16={bf16_ok} | free_disk={disk_gb:.1f} GiB')
if not bf16_ok:
    raise RuntimeError(f'Device {dev_name} does not support BF16!')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/highway_occ_flow_v26')
for req in ('code_bundle.tar.gz', 'manifest.jsonl', 'pack-sha256.json', 'stage2.pt'):
    p = DRIVE_BASE / req
    if not p.is_file():
        raise FileNotFoundError(f'Missing required file in Drive: {p}')
shards = sorted((DRIVE_BASE / 'shards').glob('*.tar'))
print(f'Drive verified: root={DRIVE_BASE} | shards={len(shards)}')
if len(shards) != 26:
    raise RuntimeError(f'Expected 26 shards in {DRIVE_BASE / "shards"}, found {len(shards)}')

needed = {
    'pytorch_lightning': 'pytorch-lightning',
    'yaml': 'pyyaml',
    'psutil': 'psutil',
    'cv2': 'opencv-python-headless',
}
missing_pkgs = []
for mod, pkg in needed.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing_pkgs.append(pkg)

if missing_pkgs:
    print('Installing missing packages:', missing_pkgs)
    if shutil.which('uv'):
        rc = subprocess.call(['uv', 'pip', 'install', '--system', *missing_pkgs])
        if rc != 0:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_pkgs])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_pkgs])

import pytorch_lightning as pl
print(f'DEPS_READY: pytorch_lightning={pl.__version__}')

In [ ]:
# %% 2. Extract Code + Verify SHA-256 + Parallel Unpack 26 Shards
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

DRIVE_BASE = Path('/content/drive/MyDrive/highway_occ_flow_v26')
REPO = Path('/content/repo')
CORPUS = Path('/content/corpus')
WEIGHTS = Path('/content/stage2.pt')

REPO.mkdir(parents=True, exist_ok=True)
(CORPUS / 'episodes').mkdir(parents=True, exist_ok=True)
(CORPUS / '.staging').mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        while chunk := f.read(4 * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

def atomic_copy_verified(src: Path, dst: Path, expected_sha: str | None = None):
    if dst.is_file() and expected_sha is not None and sha256_file(dst) == expected_sha:
        return
    tmp = dst.with_suffix(dst.suffix + '.tmp')
    h = hashlib.sha256()
    with src.open('rb') as fin, tmp.open('wb') as fout:
        while chunk := fin.read(4 * 1024 * 1024):
            h.update(chunk)
            fout.write(chunk)
    digest = h.hexdigest()
    if expected_sha is not None and digest != expected_sha:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(f'SHA-256 mismatch copying {src.name}: {digest} != {expected_sha}')
    os.replace(tmp, dst)

subprocess.check_call(['tar', 'xzf', str(DRIVE_BASE / 'code_bundle.tar.gz'), '-C', str(REPO)])
assert (REPO / 'v2_mapless_wa' / 'phase26_train.py').is_file()

pack_meta = json.loads((DRIVE_BASE / 'pack-sha256.json').read_text(encoding='utf-8'))
atomic_copy_verified(DRIVE_BASE / 'pack-sha256.json', CORPUS / 'pack-sha256.json')
atomic_copy_verified(
    DRIVE_BASE / 'manifest.jsonl',
    CORPUS / 'manifest.jsonl',
    expected_sha=pack_meta['manifest_sha256'],
)
atomic_copy_verified(DRIVE_BASE / 'stage2.pt', WEIGHTS)
print('Control files, manifest (sha256 verified), repo code, and stage2.pt copied to /content NVMe.')

files = pack_meta['files']
total_shards = len(files)

def unpack_shard(idx_and_item):
    idx, (tar_name, rec) = idx_and_item
    ep_id = rec['episode']
    dst = CORPUS / 'episodes' / ep_id
    if (dst / 'episode.json').is_file():
        return ep_id, 0, 0.0, True

    t0 = time.monotonic()
    src_tar = DRIVE_BASE / 'shards' / tar_name
    tmp_tar = Path('/tmp') / tar_name
    stg = CORPUS / '.staging' / ep_id
    shutil.rmtree(stg, ignore_errors=True)
    stg.mkdir(parents=True, exist_ok=True)

    h = hashlib.sha256()
    copied = 0
    try:
        with src_tar.open('rb') as fin, tmp_tar.open('wb') as fout:
            while chunk := fin.read(4 * 1024 * 1024):
                h.update(chunk)
                fout.write(chunk)
                copied += len(chunk)
        digest = h.hexdigest()
        if copied != int(rec['bytes']) or digest != rec['sha256']:
            raise RuntimeError(
                f'{tar_name} integrity check failed: bytes={copied}/{rec["bytes"]} sha={digest}/{rec["sha256"]}'
            )
        subprocess.check_call(['tar', 'xf', str(tmp_tar), '-C', str(stg)])
    finally:
        tmp_tar.unlink(missing_ok=True)

    extracted_ep = stg / 'episodes' / ep_id
    if not (extracted_ep / 'episode.json').is_file():
        raise RuntimeError(f'Missing episode.json after unpacking {tar_name}')
    shutil.rmtree(dst, ignore_errors=True)
    shutil.move(str(extracted_ep), str(dst))
    shutil.rmtree(stg, ignore_errors=True)
    dt = max(time.monotonic() - t0, 1e-6)
    mbps = (copied / 1e6) / dt
    print(f'[{idx}/{total_shards}] OK {ep_id} ({copied / (1 << 20):.1f} MiB in {dt:.1f}s = {mbps:.1f} MB/s)', flush=True)
    return ep_id, copied, dt, False

t_all = time.monotonic()
items = list(enumerate(sorted(files.items()), start=1))
bytes_unpacked = 0
with ThreadPoolExecutor(max_workers=4) as pool:
    futs = [pool.submit(unpack_shard, item) for item in items]
    for fut in as_completed(futs):
        _ep, nbytes, _dt, skipped = fut.result()
        if not skipped:
            bytes_unpacked += nbytes

eps_ready = sorted((CORPUS / 'episodes').glob('*/episode.json'))
print(f'UNPACK_COMPLETE: {len(eps_ready)}/{total_shards} episodes ready ({bytes_unpacked / (1 << 30):.2f} GiB in {time.monotonic() - t_all:.1f}s)')
assert len(eps_ready) == total_shards

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from v2_mapless_wa.phase26_train import Stage26ClipDataset
val_ds = Stage26ClipDataset(['/content/corpus'], 'val', clip_len=4)
print(f'VAL_CLIPS_READY: {len(val_ds)}')
assert len(val_ds) == 500

In [ ]:
# %% 3. 1-Step CUDA Flash Smoke Check + Auto Batch
import json
import os
from pathlib import Path
import shutil
import subprocess

DRIVE_RUN = Path('/content/drive/MyDrive/highway_occ_flow_v26/runs/stage26')
resuming = (DRIVE_RUN / 'checkpoints' / 'last.ckpt').is_file() and not (DRIVE_RUN / 'DONE.json').is_file()

if resuming and (DRIVE_RUN / 'training-config.json').is_file():
    prev_cfg = json.loads((DRIVE_RUN / 'training-config.json').read_text(encoding='utf-8'))
    TRAIN_BATCH = int(prev_cfg['batch'])
    TRAIN_ACCUM = int(prev_cfg['accumulate'])
    print(f'Resuming existing run with saved batch={TRAIN_BATCH}, accumulate={TRAIN_ACCUM}; skipping smoke test.')
else:
    def run_smoke(batch: int, accum: int) -> tuple[int, str]:
        for d in (Path('/content/smoke_run'), Path('/content/smoke_frozen')):
            shutil.rmtree(d, ignore_errors=True)
        cmd = [
            'python3', '-u', '-m', 'v2_mapless_wa.phase26_train',
            '--datasets', '/content/corpus',
            '--warm-start', '/content/stage2.pt',
            '--clip-len', '4',
            '--clips-per-epoch', '12000',
            '--epochs', '1',
            '--max-steps', '1',
            '--limit-val-batches', '2',
            '--batch', str(batch),
            '--accumulate', str(accum),
            '--workers', '4',
            '--sdpa', 'cuda_flash',
            '--out', '/content/smoke_run',
            '--freeze', '/content/smoke_frozen',
        ]
        env = dict(os.environ, PYTHONPATH='/content/repo')
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
        lines = []
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='', flush=True)
            lines.append(line)
        return proc.wait(), ''.join(lines)

    rc, out = run_smoke(8, 2)
    TRAIN_BATCH, TRAIN_ACCUM = 8, 2
    if rc != 0:
        low = out.lower()
        if 'out of memory' in low or 'cuda error' in low:
            print('OOM at batch=8; retrying smoke check with batch=4, accumulate=4...')
            rc, out = run_smoke(4, 4)
            TRAIN_BATCH, TRAIN_ACCUM = 4, 4
        if rc != 0:
            raise RuntimeError(f'Smoke check failed with rc={rc}')

    frozen_smoke = Path('/content/smoke_frozen/stage26.pt')
    assert frozen_smoke.is_file(), f'Missing frozen smoke weights: {frozen_smoke}'
    res = json.loads(Path('/content/smoke_run/training-result.json').read_text(encoding='utf-8'))
    print(f'SMOKE_OK: batch={TRAIN_BATCH} accumulate={TRAIN_ACCUM} result={res}')
    shutil.rmtree('/content/smoke_run', ignore_errors=True)
    shutil.rmtree('/content/smoke_frozen', ignore_errors=True)

In [ ]:
# %% 4. 10-Epoch Stage 2.6 Training + Live Drive Sync + Verify
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import threading
import time

TRAIN_BATCH = int(globals().get('TRAIN_BATCH', 8))
TRAIN_ACCUM = int(globals().get('TRAIN_ACCUM', 2))

RUN_DIR = Path('/content/stage26_run')
FROZEN_DIR = Path('/content/stage26_frozen')
DRIVE_RUN = Path('/content/drive/MyDrive/highway_occ_flow_v26/runs/stage26')

RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
FROZEN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
(DRIVE_RUN / 'checkpoints').mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        while chunk := f.read(4 * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

def atomic_copy(src: Path, dst: Path):
    if not src.is_file():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.tmp')
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

def atomic_write_json(dst: Path, obj: dict):
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2), encoding='utf-8')
    os.replace(tmp, dst)

if (DRIVE_RUN / 'DONE.json').is_file():
    print('Training already marked DONE.json on Drive; skipping re-run.')
else:
    (DRIVE_RUN / 'ERROR.json').unlink(missing_ok=True)
    resume_ckpt = None
    if (DRIVE_RUN / 'checkpoints' / 'last.ckpt').is_file():
        for name in ('training-config.json', 'validation.jsonl', 'checkpoints/best.ckpt', 'checkpoints/last.ckpt'):
            if (DRIVE_RUN / name).is_file():
                atomic_copy(DRIVE_RUN / name, RUN_DIR / name)
        resume_ckpt = RUN_DIR / 'checkpoints' / 'last.ckpt'
        print(f'Resuming training from {resume_ckpt}')

    log_path = RUN_DIR / 'train.log'
    stop_event = threading.Event()
    sync_lock = threading.Lock()

    def sync_artifacts():
        with sync_lock:
            for rel in (
                'training-config.json',
                'validation.jsonl',
                'train.log',
                'checkpoints/best.ckpt',
                'checkpoints/last.ckpt',
            ):
                src = RUN_DIR / rel
                if src.is_file():
                    try:
                        atomic_copy(src, DRIVE_RUN / rel)
                    except Exception as exc:  # noqa: BLE001
                        print(f'[sync warn] {rel}: {exc}', flush=True)

            gpu_stat = ''
            try:
                gpu_stat = subprocess.check_output(
                    ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total', '--format=csv,noheader'],
                    text=True,
                ).strip()
            except Exception:  # noqa: BLE001
                pass

            latest_val = None
            val_path = RUN_DIR / 'validation.jsonl'
            if val_path.is_file():
                try:
                    lines = [ln.strip() for ln in val_path.read_text(encoding='utf-8').splitlines() if ln.strip()]
                    if lines:
                        latest_val = json.loads(lines[-1])
                except Exception:  # noqa: BLE001
                    pass

            hb = {
                'timestamp_utc': datetime.now(timezone.utc).isoformat(),
                'batch': TRAIN_BATCH,
                'accumulate': TRAIN_ACCUM,
                'gpu': gpu_stat,
                'latest_val': latest_val,
                'best_exists': (RUN_DIR / 'checkpoints' / 'best.ckpt').is_file(),
                'last_exists': (RUN_DIR / 'checkpoints' / 'last.ckpt').is_file(),
            }
            try:
                atomic_write_json(DRIVE_RUN / 'heartbeat.json', hb)
            except Exception as exc:  # noqa: BLE001
                print(f'[sync warn] heartbeat: {exc}', flush=True)

    def sync_loop():
        while not stop_event.wait(60):
            sync_artifacts()

    sync_thread = threading.Thread(target=sync_loop, daemon=True)
    sync_thread.start()

    cmd = [
        'python3', '-u', '-m', 'v2_mapless_wa.phase26_train',
        '--datasets', '/content/corpus',
        '--warm-start', '/content/stage2.pt',
        '--clip-len', '4',
        '--clips-per-epoch', '12000',
        '--epochs', '10',
        '--batch', str(TRAIN_BATCH),
        '--accumulate', str(TRAIN_ACCUM),
        '--workers', '6',
        '--lr', '2e-4',
        '--sdpa', 'cuda_flash',
        '--seed', '20261121',
        '--out', str(RUN_DIR),
        '--freeze', str(FROZEN_DIR),
    ]
    if resume_ckpt is not None:
        cmd.extend(['--resume', str(resume_ckpt)])

    rc = -1
    tail = []
    try:
        env = dict(os.environ, PYTHONPATH='/content/repo')
        with log_path.open('w', encoding='utf-8', buffering=1) as log_f:
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
            assert proc.stdout is not None
            for line in proc.stdout:
                print(line, end='', flush=True)
                log_f.write(line)
                tail.append(line)
                if len(tail) > 80:
                    tail.pop(0)
            rc = proc.wait()
        if rc != 0:
            raise RuntimeError(f'phase26_train exited with rc={rc}')

        stop_event.set()
        sync_thread.join(timeout=120)
        sync_artifacts()

        local_pt = FROZEN_DIR / 'stage26.pt'
        local_meta = FROZEN_DIR / 'metadata.json'
        local_res = RUN_DIR / 'training-result.json'
        meta = json.loads(local_meta.read_text(encoding='utf-8'))
        res = json.loads(local_res.read_text(encoding='utf-8'))

        atomic_copy(local_pt, DRIVE_RUN / 'stage26.pt')
        atomic_copy(local_meta, DRIVE_RUN / 'metadata.json')
        atomic_copy(local_res, DRIVE_RUN / 'training-result.json')

        sha_local = sha256_file(local_pt)
        sha_drive = sha256_file(DRIVE_RUN / 'stage26.pt')
        if sha_local != meta['sha256'] or sha_drive != meta['sha256']:
            raise RuntimeError(f'Frozen weight SHA mismatch: local={sha_local} drive={sha_drive} meta={meta["sha256"]}')

        done_payload = {
            'status': 'ok',
            'timestamp_utc': datetime.now(timezone.utc).isoformat(),
            'sha256': sha_local,
            'batch': TRAIN_BATCH,
            'accumulate': TRAIN_ACCUM,
            'metadata': meta,
            'training_result': res,
        }
        atomic_write_json(DRIVE_RUN / 'DONE.json', done_payload)
        os.sync()
        time.sleep(15)
        print('TRAINING_DONE_VERIFIED:', json.dumps(done_payload, indent=2))
    except Exception as exc:  # noqa: BLE001
        stop_event.set()
        sync_thread.join(timeout=120)
        sync_artifacts()
        err_payload = {
            'status': 'error',
            'timestamp_utc': datetime.now(timezone.utc).isoformat(),
            'returncode': rc,
            'error': repr(exc),
            'batch': TRAIN_BATCH,
            'accumulate': TRAIN_ACCUM,
            'log_tail': ''.join(tail[-40:]),
        }
        atomic_write_json(DRIVE_RUN / 'ERROR.json', err_payload)
        os.sync()
        time.sleep(10)
        raise

In [ ]:
# %% 5. Auto-Release L4 Runtime
import time
from pathlib import Path
from google.colab import runtime

done_file = Path('/content/drive/MyDrive/highway_occ_flow_v26/runs/stage26/DONE.json')
if not done_file.is_file():
    raise RuntimeError(f'Refusing to auto-shutdown without DONE.json at {done_file}')

print(f'Verified {done_file} exists. Waiting 10s for final Drive flush before releasing L4 runtime...')
time.sleep(10)
print('Releasing Colab runtime now (runtime.unassign()).')
runtime.unassign()